# Exercise 61 — Reshaping: pivot, melt

## Concept

**Long format** vs **wide format**:

```
Long (tidy)                        Wide
────────────────                   ──────────────────────────
date        region  amount         date        North  South  East
2026-01-01  North   100            2026-01-01    100     80    50
2026-01-01  South    80            2026-01-02    150    NaN    70
2026-01-01  East     50
2026-01-02  North   150
2026-01-02  East     70
```

Long is what databases prefer (one fact per row). Wide is what humans like to read and what spreadsheets/reports use.

### Long → wide: `pivot_table`

```python
df.pivot_table(
    index="date",
    columns="region",
    values="amount",
    aggfunc="sum",      # required if duplicate (date, region) pairs exist
    fill_value=0,       # replace NaN with 0
)
```

`pivot_table` handles duplicates via the aggregation function. There's also `.pivot(...)` which is stricter and raises on duplicates — usually `pivot_table` is what you want.

### Wide → long: `melt`

```python
df.melt(
    id_vars=["date"],                          # columns to keep as-is
    value_vars=["North", "South", "East"],     # columns to unpivot
    var_name="region",                         # new column name for the old column headers
    value_name="amount",                       # new column name for the values
)
```

### When to reshape

- For aggregations and joins: keep data **long**. One observation per row.
- For a final report: pivot to **wide** at the end.
- Reshape early in the pipeline only if absolutely necessary — most operations work better on long data.

## Setup

In [ ]:
import pandas as pd

long_df = pd.DataFrame({
    "date":   ["2026-01-01"] * 3 + ["2026-01-02"] * 2,
    "region": ["North", "South", "East", "North", "East"],
    "amount": [100, 80, 50, 150, 70],
})
print("long:\n", long_df)


## Task 1 — Long → wide

Pivot `long_df` so that:
- `date` is the row index
- `region` values become column names
- `amount` fills the cells
- Missing (date, region) combinations become `0`

Return a DataFrame where `date` is **a column** (not the index — `.reset_index()`).

```python
def long_to_wide(df: pd.DataFrame) -> pd.DataFrame:
    ...
```

Expected:
```
         date  East  North  South
0  2026-01-01    50    100     80
1  2026-01-02    70    150      0
```

In [ ]:
import pandas as pd

def long_to_wide(df: pd.DataFrame) -> pd.DataFrame:
    pass  # TODO

result = long_to_wide(long_df)
# pivot_table sorts columns alphabetically by default → East, North, South
assert "date" in result.columns
assert set(result.columns) >= {"date", "East", "North", "South"}
row0 = result[result["date"] == "2026-01-01"].iloc[0]
assert row0["East"] == 50 and row0["North"] == 100 and row0["South"] == 80
row1 = result[result["date"] == "2026-01-02"].iloc[0]
assert row1["East"] == 70 and row1["North"] == 150 and row1["South"] == 0
print("ok")


## Task 2 — Wide → long

Take a wide DataFrame and melt it back to long format with columns `["date", "region", "amount"]`. Drop rows where `amount` is 0 (treat 0 as "this region/date combo doesn't exist" — a common cleanup after a `fill_value=0` pivot).

```python
def wide_to_long(wide: pd.DataFrame) -> pd.DataFrame:
    ...
```

The wide fixture is the output of Task 1. Expected long output has 5 rows (the 0 cell is dropped).

In [ ]:
import pandas as pd

def wide_to_long(wide: pd.DataFrame) -> pd.DataFrame:
    pass  # TODO

wide = long_to_wide(long_df)
result = wide_to_long(wide).sort_values(["date", "region"]).reset_index(drop=True)
assert list(result.columns) == ["date", "region", "amount"]
assert len(result) == 5
assert (result["amount"] != 0).all()
print("ok")


## Task 3 — Pivot with multiple aggregations

Add a `qty` column to the long data. Then pivot to produce **both** sum of `amount` and sum of `qty` per (date, region). The resulting columns should be a `MultiIndex` like `('amount', 'North')`.

```python
def pivot_amount_and_qty(df: pd.DataFrame) -> pd.DataFrame:
    ...
```

Then flatten the column index to `"amount_North"`, `"qty_North"`, etc., so the result has plain string column names. Reset index so `date` is a column.

In [ ]:
import pandas as pd

long_qty = long_df.assign(qty=[2, 1, 1, 3, 2])
print(long_qty)

def pivot_amount_and_qty(df: pd.DataFrame) -> pd.DataFrame:
    pass  # TODO

result = pivot_amount_and_qty(long_qty)
assert "date" in result.columns
flat_cols = [c for c in result.columns if c != "date"]
# Expect 6: amount_East, amount_North, amount_South, qty_East, qty_North, qty_South
assert set(flat_cols) == {"amount_East", "amount_North", "amount_South", "qty_East", "qty_North", "qty_South"}
row0 = result[result["date"] == "2026-01-01"].iloc[0]
assert row0["amount_North"] == 100
assert row0["qty_North"] == 2
print("ok")
